## 1. Setup

In [150]:
import pandas as pd
from pathlib import Path

SIMULATED_DIR = Path("../data/simulated")

## 2. Load the data

In [151]:
ehr_raw = pd.read_csv(SIMULATED_DIR / "ehr_providers.csv")
hr_raw = pd.read_csv(SIMULATED_DIR / "hr_providers.csv")
cred_raw = pd.read_csv(SIMULATED_DIR / "credentialing_providers.csv")

## 3. Set up findings log and helper functions

In [152]:
findings = pd.DataFrame(columns=[
    "dataset",
    "column",
    "issue",
    "example",
    "planned_action",    
    "status"
])

# Helper function to add log
def add_finding(dataset, column, issue, example, planned_action):
    duplicate = (
        (findings["dataset"] == dataset) &
        (findings["column"] == column) &
        (findings["issue"] == issue)
    ).any()
    
    if duplicate:
        print("Finding already exists.")
        return
    
    findings.loc[len(findings)] = {
        "dataset": dataset,
        "column": column,
        "issue": issue,
        "example": example,
        "planned_action": planned_action,
        "status": "Open"
    }

# Helper function to delete log
def delete_finding(dataset, column, issue):
    global findings
    
    findings = findings[
        ~(
            (findings["dataset"] == dataset) &
            (findings["column"] == column) &
            (findings["issue"] == issue)
        )
    ].reset_index(drop=True)

## 4. Inspecting the datasets

In [153]:
simulated_providers = {
    "EHR" : ehr_raw,
    "HR" : hr_raw,
    "CREDENTIALING" : cred_raw
}

# Checking columns and df shape
for name, df in simulated_providers.items():
    print(name)
    print(df.shape)
    print(df.columns)
    print('\n')

EHR
(3000, 13)
Index(['ehr_provider_id', 'npi', 'first_name', 'middle_name', 'last_name',
       'credential', 'specialty_code', 'address_line_1', 'address_line_2',
       'city', 'state', 'zip', 'phone'],
      dtype='str')


HR
(2250, 10)
Index(['employee_id', 'npi', 'first_name', 'middle_initial', 'last_name',
       'job_credential', 'job_specialty_code', 'work_city', 'work_state',
       'work_phone'],
      dtype='str')


CREDENTIALING
(2550, 15)
Index(['credentialing_id', 'npi', 'legal_first_name', 'legal_middle_name',
       'legal_last_name', 'credential', 'taxonomy_code', 'license_number',
       'license_state', 'address_line_1', 'address_line_2', 'city', 'state',
       'zip', 'phone'],
      dtype='str')




## 5. Validating column datatypes

In [154]:
for name, df in simulated_providers.items():
    print(name)
    print(df.dtypes)
    print('\n')

# NPI stored as float across EHR, HR, and CREDENTIALING
# Convert to string and validate 10 digits
for dataset in ["EHR", "HR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="npi",
        issue="NPI stored as float",
        example="1234567890.0",
        planned_action="Convert to string and validate 10 digits"
    )

# Zip stored as int across EHR and CREDENTIALING
# Convert to string 
for dataset in ["EHR", "Credentialing"]:
    add_finding(
        dataset=dataset,
        column="zip",
        issue="Zip stored as int",
        example="773386118",
        planned_action="Convert to string"
    )

EHR
ehr_provider_id        str
npi                float64
first_name             str
middle_name            str
last_name              str
credential             str
specialty_code         str
address_line_1         str
address_line_2         str
city                   str
state                  str
zip                  int64
phone                  str
dtype: object


HR
employee_id               str
npi                   float64
first_name                str
middle_initial            str
last_name                 str
job_credential            str
job_specialty_code        str
work_city                 str
work_state                str
work_phone                str
dtype: object


CREDENTIALING
credentialing_id         str
npi                  float64
legal_first_name         str
legal_middle_name        str
legal_last_name          str
credential               str
taxonomy_code            str
license_number           str
license_state            str
address_line_1           str
addres

In [155]:
cred_raw.head()

# Phone stored as float in CREDENTIALING
# Convert to string and validate
add_finding(
        dataset="Credentialing",
        column="phone",
        issue="phone stored as float",
        example="2.816416e+09",
        planned_action="Convert to string and validate length"
    )

## 6. Check for missingness

In [156]:
for name, df in simulated_providers.items():
    print(f"\n{name} Missingness")
    
    missing_summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })
    
    display(
        missing_summary[
            missing_summary["missing_count"] > 0
        ].sort_values("missing_pct", ascending=False)
    )


EHR Missingness


,missing_count,missing_pct
address_line_2,2667,88.90
middle_name,1468,48.93
credential,836,27.87
npi,238,7.93
phone,156,5.20



HR Missingness


,missing_count,missing_pct
npi,1206,53.60
middle_initial,1161,51.60
job_credential,620,27.56
job_specialty_code,481,21.38
work_phone,279,12.40



CREDENTIALING Missingness


,missing_count,missing_pct
address_line_2,2201,86.31
legal_middle_name,1170,45.88
credential,708,27.76
license_number,421,16.51
license_state,371,14.55
phone,191,7.49
npi,51,2.00


## 7. Check for duplicates

In [157]:
# No duplicated rows found
for name, df in simulated_providers.items():
    print(df[df.duplicated(keep=False)])

Empty DataFrame
Columns: [ehr_provider_id, npi, first_name, middle_name, last_name, credential, specialty_code, address_line_1, address_line_2, city, state, zip, phone]
Index: []
Empty DataFrame
Columns: [employee_id, npi, first_name, middle_initial, last_name, job_credential, job_specialty_code, work_city, work_state, work_phone]
Index: []
Empty DataFrame
Columns: [credentialing_id, npi, legal_first_name, legal_middle_name, legal_last_name, credential, taxonomy_code, license_number, license_state, address_line_1, address_line_2, city, state, zip, phone]
Index: []


In [158]:
for name, df in simulated_providers.items():
    print(df.columns)

Index(['ehr_provider_id', 'npi', 'first_name', 'middle_name', 'last_name',
       'credential', 'specialty_code', 'address_line_1', 'address_line_2',
       'city', 'state', 'zip', 'phone'],
      dtype='str')
Index(['employee_id', 'npi', 'first_name', 'middle_initial', 'last_name',
       'job_credential', 'job_specialty_code', 'work_city', 'work_state',
       'work_phone'],
      dtype='str')
Index(['credentialing_id', 'npi', 'legal_first_name', 'legal_middle_name',
       'legal_last_name', 'credential', 'taxonomy_code', 'license_number',
       'license_state', 'address_line_1', 'address_line_2', 'city', 'state',
       'zip', 'phone'],
      dtype='str')


## 8. Duplicates and missingness check for each data source

### EHR

In [159]:
ehr_raw[ehr_raw["ehr_provider_id"].duplicated(keep=False)].sort_values("ehr_provider_id")
ehr_raw[ehr_raw["npi"].duplicated(keep=False)].sort_values("npi")

# Phone contains duplicates
ehr_raw[ehr_raw["phone"].duplicated(keep=False)].sort_values("phone")

,ehr_provider_id,npi,first_name,middle_name,last_name,credential,specialty_code,address_line_1,address_line_2,city,state,zip,phone
2318,EHR002319,1.295667e+09,Odelee,A,Seda,NaN,106S00000X,903 PROTON RD,NaN,SAN ANTONIO,TX,782584203,(210) 228-9923
783,EHR000784,NaN,KATHERINE,NaN,STROBEL,RBT,106S00000X,903 PROTON RD,NaN,SAN ANTONIO,TX,782584203,(210) 228-9923
2202,EHR002203,1.639158e+09,Jeffrey,CARL,DARIUS,CRNA,367500000X,"2200 BERGQUIST DR, SUITE 1",ATTN: CREDENTIALS (CMC),LACKLAND AFB,TX,782365300,(210) 292-6707
2791,EHR002792,1.467432e+09,Don,A,LAWRENCE,D.O,207Q00000X,2200 BERGQUIST DR,ATTN: CREDENTIALS (CMC),LACKLAND A F B,TX,782369908,(210) 292-6707
2815,EHR002816,1.710232e+09,Amanda,NaN,Aranda,M.S. CCC/SLP,235Z00000X,10609 IH 10 W,201,SAN ANTONIO,TX,782301672,(210) 344-5437
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2877,EHR002878,1.801096e+09,LISA,NaN,SLAUGHTER,NaN,133N00000X,5900 BALCONES DR STE 100,NaN,AUSTIN,TX,787314298,NaN
2878,EHR002879,1.497799e+09,ROLAND,P,VARGAS,DDS,1223G0001X,6750 TEZEL RD,NaN,SAN ANTONIO,TX,782504183,NaN
2889,EHR002890,NaN,Joel,NaN,PATTERSON,M.D.,207T00000X,301 UNIVERSITY BLVD,NaN,GALVESTON,TX,775551022,NaN
2894,EHR002895,1.295672e+09,VICTORIA,NaN,GARCIA,NaN,106S00000X,4234 WEBER RD,NaN,CORPUS CHRISTI,TX,784113603,NaN


In [160]:
# inspecting duplciate phone records
phone_counts = ehr_raw["phone"].value_counts()
display(phone_counts[phone_counts > 1])

# Phone numbers are shared across multiple records
# Retain values and use as supporting match attribute only
add_finding(
    dataset="EHR",
    column="phone",
    issue="Phone numbers are shared across multiple provider records",
    example="(210) 617-5300 appears on 9 records",
    planned_action="Retain values and use as supporting match attribute only"
)

phone
(210) 617-5300    9
(855) 223-7123    9
(713) 792-6161    9
(972) 715-5000    8
(713) 620-4000    7
                 ..
(254) 732-2262    2
(210) 397-8500    2
8774182978        2
(469) 204-2021    2
9566657049        2
Name: count, Length: 122, dtype: int64

In [161]:
ehr_raw[ehr_raw["zip"].duplicated(keep=False)].sort_values("zip")

zip_counts_ehr = ehr_raw["zip"].value_counts()
display(zip_counts_ehr[zip_counts_ehr > 1])

# Zip codes are shared across multiple records
# Retain values and use as supporting match attribute only
add_finding(
    dataset="EHR",
    column="zip",
    issue="Zip codes are shared across multiple provider records",
    example="782294402 appears on 15 records",
    planned_action="Retain values and use as supporting match attribute only"
)

zip
782294402    15
782344504    14
782294404    13
753907201    13
770301501    12
             ..
770302777     2
780068546     2
785392909     2
770302761     2
782369908     2
Name: count, Length: 226, dtype: int64

In [162]:
ehr_raw["zip"].dropna().astype(str).str.len().value_counts()

# Zip has mixed 5-digit and 9-digit formats
# Standardize ZIP format and derive 5-digit ZIP for matching
add_finding(
    dataset='EHR',
    column="zip",
    issue="ZIP has mixed 5-digit and 9-digit formats",
    example="77380, 774336767",
    planned_action="Standardize ZIP format and derive 5-digit ZIP for matching"
)

### HR

In [163]:
# inspecting duplciate phone records
hr_raw.head()

,employee_id,npi,first_name,middle_initial,last_name,job_credential,job_specialty_code,work_city,work_state,work_phone
0,EMP000001,1.083934e+09,JOHANNA,NaN,RAMIREZ,MOTR,225X00000X,SAN ANTONIO,TX,800-944-9782
1,EMP000002,NaN,ESEER,NaN,AL RUKABI,RPH,NaN,AUSTIN,TX,512-459-8308
2,EMP000003,NaN,PAMELA,K,HANCOCK,LPC,101YP2500X,DENTON,TX,NaN
3,EMP000004,1.699706e+09,RYAN,NaN,BLISS,PT,NaN,BASTROP,TX,512-303-1116
4,EMP000005,NaN,MORGAN,NaN,STURGEON,NaN,NaN,KILLEEN,TX,254-554-1466


In [164]:
hr_raw[hr_raw["npi"].duplicated(keep=False)].sort_values("npi")
hr_raw[hr_raw["employee_id"].duplicated(keep=False)].sort_values("employee_id")

,employee_id,npi,first_name,middle_initial,last_name,job_credential,job_specialty_code,work_city,work_state,work_phone


In [165]:
phone_counts = hr_raw["work_phone"].value_counts()
display(phone_counts[phone_counts > 1])

# Phone numbers are shared across multiple records
# Retain values and use as supporting match attribute only
add_finding(
    dataset="HR",
    column="work_phone",
    issue="Phone numbers are shared across multiple provider records",
    example="713-792-6161 appears on 9 records",
    planned_action="Retain values and use as supporting match attribute only"
)

work_phone
713-792-6161    9
210-617-5300    7
855-223-7123    7
817-335-3022    6
972-715-5000    6
               ..
817-702-1244    2
855-984-5121    2
7137911414      2
512-693-7045    2
888-804-3000    2
Name: count, Length: 74, dtype: int64

### CREDENTIALING

In [166]:
cred_raw.head()

,credentialing_id,npi,legal_first_name,legal_middle_name,legal_last_name,credential,taxonomy_code,license_number,license_state,address_line_1,address_line_2,city,state,zip,phone
0,CRED000001,1.205445e+09,KEEGAN,NaN,MATTOX,PHARMD,1835P0018X,65602,TX,160 N COIT RD,NaN,RICHARDSON,TX,750805454,NaN
1,CRED000002,1.821717e+09,ALEXANDRA,MARIE,GARZA,NaN,235Z00000X,NaN,NaN,1700 WILSON RD,NaN,HUMBLE,TX,773386118,2.816416e+09
2,CRED000003,1.558891e+09,THEODORA,EFROSENI,TSAKALAKIS,ATC,2255A2300X,2000027672,TX,15722 DUNMOOR DR.,NaN,HOUSTON,TX,77059,NaN
3,CRED000004,1.215255e+09,RYAN,SAMUEL,STEPINOFF,PA-C,363AM0700X,17487375,TX,6913 CAMP BOWIE BLVD STE 141,NaN,FORT WORTH,TX,761167165,8.175605e+09
4,CRED000005,1.396886e+09,JOHN,EMERSON,WINEMAN,PA-C,363A00000X,PA02729,TX,120 WOOD AVE,NaN,WOODSBORO,TX,78393,3.615435e+09


In [167]:
cred_raw[cred_raw["npi"].duplicated(keep=False)].sort_values("npi")
cred_raw[cred_raw["credentialing_id"].duplicated(keep=False)].sort_values("credentialing_id")
cred_raw[cred_raw["license_number"].duplicated(keep=False)].sort_values("license_number")
cred_raw[cred_raw["zip"].duplicated(keep=False)].sort_values("zip")


,credentialing_id,npi,legal_first_name,legal_middle_name,legal_last_name,credential,taxonomy_code,license_number,license_state,address_line_1,address_line_2,city,state,zip,phone
431,CRED000432,1.306969e+09,PATRICK,L,ADUDDELL,DDS,1223G0001X,14146,TX,1300 MEDICAL AVE,#101,PLANO,TX,75075,9.728676e+09
1934,CRED001935,1.972715e+09,PATRICIA,ELLEN,STANCLIFF,PHYSICIAL THERAPIST,225100000X,1026706,TX,1201 BALBOA CIRCLE,NaN,PLANO,TX,75075,9.724232e+09
2293,CRED002294,1.225136e+09,SHELTON,K,JONES,PA-C,363AS0400X,PA04684,TX,6020 W PARKER RD STE 470,NaN,PLANO,TX,75093,9.726089e+09
1422,CRED001423,1.053349e+09,SUREKHA,NaN,PERLMAN,M.D.,207RE0101X,K6230,TX,6124 W PARKER RD,SUITE 332,PLANO,TX,75093,9.729818e+09
843,CRED000844,1.649287e+09,ELIZABETH,R,VAUGHAN,MD,207W00000X,D3425,TX,2811 LEMMON AVE E,STE 202,DALLAS,TX,75204,2.145226e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,CRED000910,1.326053e+09,HOMER,J,LEMAR,M.D.,207RE0101X,04-20304,KS,5005 N PIEDRAS ST,NaN,EL PASO,TX,799205001,9.155692e+09
1052,CRED001053,1.972201e+09,CRISTINA,IVETTE,RODRIGUEZ,NP,363LF0000X,1110944,TX,5001 N PIEDRAS ST,NaN,EL PASO,TX,799304210,NaN
1662,CRED001663,1.497366e+09,BERI,J,GLOVER,AUD,231H00000X,2020022062,MO,5001 N PIEDRAS ST,NaN,EL PASO,TX,799304210,9.155646e+09
623,CRED000624,1.417967e+09,ROBERT,L.,GREEN,PA,363A00000X,PA04506,TX,1400 GEORGE DIETER DR,SUITE 100,EL PASO,TX,799367601,9.158575e+09


In [168]:
zip_counts_cred = cred_raw["zip"].value_counts()
display(zip_counts_cred[zip_counts_cred > 1])

# Zip codes are shared across multiple records
# Retain values and use as supporting match attribute only
add_finding(
    dataset="Credentialing",
    column="zip",
    issue="Zip codes are shared across multiple provider records",
    example="782344504 appears on 13 records",
    planned_action="Retain values and use as supporting match attribute only"
)

zip
782344504    13
753907201    13
770301501    12
770304000    11
782294402    11
             ..
780415955     2
760537209     2
765485725     2
760865705     2
785031241     2
Name: count, Length: 189, dtype: int64

## 9. Formatting check for each data source

### EHR

In [173]:
# Phone numbers exist in varied formats
# Retain values and use as supporting match attribute only
add_finding(
    dataset="EHR",
    column="phone",
    issue="Phone numbers exist in varied formats",
    example="(210) 617-5300, 8774182978",
    planned_action="Standardise phone formats",
)

Finding already exists.


### HR

In [170]:
hr_raw["work_phone"].dropna().astype(str).str.len().value_counts()

# Phone numbers exist in varied formats
# Inspect and standardise phone formats
add_finding(
    dataset="HR",
    column="work_phone",
    issue="Phone numbers exist in varied formats",
    example="817-702-1244, 7137911414",
    planned_action="Inspect and standardise phone formats"
)

### CREDENTIALING

In [171]:
cred_raw["zip"].dropna().astype(str).str.len().value_counts()

# Zip has mixed 5-digit and 9-digit formats
# Standardize ZIP format and derive 5-digit ZIP for matching
add_finding(
    dataset='Credentialing',
    column="zip",
    issue="ZIP has mixed 5-digit and 9-digit formats",
    example="77380, 774336767",
    planned_action="Standardize ZIP format and derive 5-digit ZIP for matching"
)

In [174]:
findings

,dataset,column,issue,example,planned_action,status
0,EHR,npi,NPI stored as float,1234567890.0,Convert to string and validate 10 digits,Open
1,HR,npi,NPI stored as float,1234567890.0,Convert to string and validate 10 digits,Open
2,Credentialing,npi,NPI stored as float,1234567890.0,Convert to string and validate 10 digits,Open
3,EHR,zip,Zip stored as int,773386118,Convert to string,Open
4,Credentialing,zip,Zip stored as int,773386118,Convert to string,Open
5,Credentialing,phone,phone stored as float,2.816416e+09,Convert to string and validate length,Open
6,EHR,phone,Phone numbers are shared across multiple provi...,(210) 617-5300 appears on 9 records,Retain values and use as supporting match attr...,Open
7,EHR,zip,Zip codes are shared across multiple provider ...,782294402 appears on 15 records,Retain values and use as supporting match attr...,Open
8,EHR,zip,ZIP has mixed 5-digit and 9-digit formats,"77380, 774336767",Standardize ZIP format and derive 5-digit ZIP ...,Open
9,HR,work_phone,Phone numbers are shared across multiple provi...,713-792-6161 appears on 9 records,Retain values and use as supporting match attr...,Open
